# TP15 - Méthodes d'ensemble

**Objectif :** pousser la performance proprement et savoir dire ce que le modèle rate.

## 1. Remettre le décor en place

Chargement des données

In [2]:
import pandas as pd

df = pd.read_parquet("../data/ml/nutriscope_ml_sample.parquet")
display(df.shape)
display(df["split"].value_counts())
display(df["nutriscore_grade"].value_counts(normalize=True))
display(df)

(30041, 21)

split
train    24076
test      5965
Name: count, dtype: int64

nutriscore_grade
e    0.286409
d    0.264372
c    0.207550
a    0.134150
b    0.107520
Name: proportion, dtype: float64

,code,product_name,brands,pnns_groups_1,pnns_groups_2,nova_group,additives_n,energy-kcal_100g,fat_100g,saturated-fat_100g,...,sugars_100g,fiber_100g,proteins_100g,salt_100g,fruits-vegetables-legumes_100g,nutriscore_score,nutriscore_grade,completeness,last_modified,split
0,0000112302660,Soupe potiron et kiri,Liebig,Fruits and vegetables,Soups,4,4,38.300000,1.50000,1.000000,...,2.000000,0.800000,0.90000,0.740000,34.312500,3,c,0.6875,2026-03-14,train
1,0000112302705,velouté de tomates et pointe d'origan,Liebig,Fruits and vegetables,Soups,4,0,28.000000,0.50000,0.300000,...,2.400000,1.000000,0.70000,0.670000,72.977007,1,b,0.7625,2026-03-19,test
2,0000112302754,pur soup' mouliné de 5 légumes verts,liebig,Composite foods,One-dish meals,3,0,24.802376,0.16035,0.044483,...,0.738538,0.837462,0.81314,0.035146,23.242188,0,a,0.9750,2026-03-26,train
3,0000112307261,Velouté de 10 légumes,Liebig,Fruits and vegetables,Soups,4,0,43.000000,1.60000,0.600000,...,1.300000,0.800000,0.70000,0.670000,34.981336,3,c,0.9000,2026-03-26,train
4,0000300000004,Lentilles Brunes Bio,Kazidomi,Composite foods,One-dish meals,3,0,54.000000,0.30000,0.000000,...,0.000000,4.000000,4.00000,0.080000,100.000000,-7,a,0.8875,2026-03-17,test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30036,9950601010759,Graine de lin brun,NaN,Cereals and potatoes,Cereals,<NA>,<NA>,527.000000,43.00000,4.300000,...,0.900000,NaN,20.10000,0.120000,NaN,-5,b,0.3625,2026-03-24,test
30037,99623456,Petales de patates douces,Croustisud,Salty snacks,Appetizers,3,0,538.000000,35.10000,4.000000,...,27.000000,1.865500,4.30000,0.300000,0.000000,17,d,0.5750,2026-03-20,train
30038,99895488,Langouste cuite coupée en 2,NaN,Fish Meat Eggs,Fish and seafood,<NA>,<NA>,136.000000,1.94000,0.297000,...,0.000000,NaN,26.40000,0.500000,NaN,-4,a,0.6750,2026-03-20,train
30039,99993306,Pizza thon cuite sur pierre,Trattoria Alfredo,Composite foods,Pizza pies and quiches,<NA>,<NA>,231.000000,9.80000,3.500000,...,5.500000,NaN,10.00000,1.230000,NaN,12,d,0.4750,2026-03-20,test


Choix des colonnes

In [3]:
NUTRIMENTS = [
    "energy-kcal_100g", "fat_100g", "saturated-fat_100g", "carbohydrates_100g",
    "sugars_100g", "proteins_100g", "salt_100g"
]
OPTIONNELLES = [
    "fiber_100g", "fruits-vegetables-legumes_100g", "nova_group", "additives_n"
]
CATEGORIELLES = ["pnns_groups_1", "pnns_groups_2"]

FEATURES = NUTRIMENTS + OPTIONNELLES + CATEGORIELLES

FORBIDDEN = ["nutriscore_score", "code", "product_name", "brands", "split"]

for col in FORBIDDEN:
    assert col not in FEATURES, f"{col} ne doit pas être une feature"

Préservation du train

In [4]:
from sklearn.preprocessing import LabelEncoder

train = df[df["split"] == "train"]

X_train = train[FEATURES]

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train["nutriscore_grade"])
print(label_encoder.classes_)

['a' 'b' 'c' 'd' 'e']


Définition des règles

In [5]:
from sklearn.model_selection import StratifiedKFold, cross_validate


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

SCORING = ["f1_macro", "accuracy", "balanced_accuracy"]

def evaluate_cv(name: str, model, X: pd.DataFrame) -> dict:
    """Évalue un modèle par validation croisée sur le train, avec les plis et métriques communs.

    Tous les modèles passent par cette fonction : ils sont donc comparés sur les
    mêmes plis (`cv`) et les mêmes métriques (`SCORING`), condition d'un tableau
    comparatif valable.

    Args:
        name: nom du modèle, affiché dans le tableau des résultats.
        model: modèle ou Pipeline scikit-learn, non entraîné.
        X: features du train (la cible utilisée est `y_train`).

    Returns:
        Dictionnaire avec la moyenne des scores sur les 5 plis, l'écart-type du
        F1 macro, et les temps moyens d'ajustement et de prédiction (en secondes).
    """
    r = cross_validate(model, X, y_train, cv=cv, scoring=SCORING)
    return {
        "model": name,
        "f1_macro": r["test_f1_macro"].mean(),
        "f1_std": r["test_f1_macro"].std(),
        "accuracy": r["test_accuracy"].mean(),
        "balanced_accuracy": r["test_balanced_accuracy"].mean(),
        "fit_time": r["fit_time"].mean(), "score_time": r["score_time"].mean()
    }

results = []


Préparation et entraînement

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

import numpy as np

def add_derived_features(X: pd.DataFrame) -> pd.DataFrame:
    """Ajoute deux ratios et l'écart d'énergie (règle 4/4/9 du TP 9)."""
    X = X.copy()
    X["sat_fat_ratio"] = (X["saturated-fat_100g"] / X["fat_100g"].replace(0, np.nan)).fillna(0)
    X["sugar_ratio"] = (X["sugars_100g"] / X["carbohydrates_100g"].replace(0, np.nan)).fillna(0)
    energy_calc = 4 * X["carbohydrates_100g"] + 4 * X["proteins_100g"] + 9 * X["fat_100g"]
    X["energy_gap"] = X["energy-kcal_100g"] - energy_calc
    return X

DERIVEES = ["sat_fat_ratio", "sugar_ratio", "energy_gap"]


optional_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("nutr", StandardScaler(), NUTRIMENTS + DERIVEES),
    ("opt", optional_pipeline, OPTIONNELLES),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=80), CATEGORIELLES)
])

prep_pipeline = Pipeline([
    ("derived", FunctionTransformer(add_derived_features)),
    ("prep", preprocessor),
])

In [7]:
from sklearn.linear_model import LogisticRegression


logreg = Pipeline([("prep", prep_pipeline), ("model", LogisticRegression(max_iter=3000))])
results.append(evaluate_cv("logreg", logreg, X_train))

display(pd.DataFrame(results).set_index("model").round(3))

,f1_macro,f1_std,accuracy,balanced_accuracy,fit_time,score_time
model,,,,,,
logreg,0.767,0.008,0.805,0.766,0.727,0.017


In [8]:
from sklearn.ensemble import RandomForestClassifier


rnd_forest = Pipeline([("prep", prep_pipeline), ("model", RandomForestClassifier(random_state=42))])
results.append(evaluate_cv("rnd_forest", rnd_forest, X_train))

display(pd.DataFrame(results).set_index("model").round(3))

,f1_macro,f1_std,accuracy,balanced_accuracy,fit_time,score_time
model,,,,,,
logreg,0.767,0.008,0.805,0.766,0.727,0.017
rnd_forest,0.858,0.007,0.877,0.854,1.925,0.079


## 2. Boosting

In [9]:
from sklearn.ensemble import HistGradientBoostingClassifier

hist_gb = Pipeline([("prep", prep_pipeline), ("model", HistGradientBoostingClassifier(random_state=42))])
results.append(evaluate_cv("hist_gb", hist_gb, X_train))

In [10]:
from xgboost import XGBClassifier

xg_boost = Pipeline([("prep", prep_pipeline), ("model", XGBClassifier(random_state=42))])
results.append(evaluate_cv("xg_boost", xg_boost, X_train))


In [11]:
from lightgbm import LGBMClassifier

light_gbm = Pipeline([("prep", prep_pipeline), ("model", LGBMClassifier(random_state=42, verbose=-1))])
results.append(evaluate_cv("light_gbm", light_gbm, X_train))

display(pd.DataFrame(results).set_index("model").round(3))


,f1_macro,f1_std,accuracy,balanced_accuracy,fit_time,score_time
model,,,,,,
logreg,0.767,0.008,0.805,0.766,0.727,0.017
rnd_forest,0.858,0.007,0.877,0.854,1.925,0.079
hist_gb,0.892,0.006,0.910,0.891,2.203,0.051
xg_boost,0.894,0.005,0.910,0.892,1.122,0.034
light_gbm,0.888,0.007,0.906,0.887,0.773,0.051


Deux choses notables :
- Le boosting bat la forêt avec un écart de +0.03, soit quatre à cinq fois l'écart-type. On a donc un gain réel.
- Les trois boosters sont à égalité : il n'y a que 0.006 d'écart, ce qui est de l'ordre d'un écart-type ; les trois boosters étant indiscernables en F1 macro, ce seront les autres critères (temps, taille, simplicité) qui les départageront.

Choix du booster : LightGBM
Les trois boosters sont indiscernables en F1 macro et accuracy, mais LightGBM est 1.6 fois plus rapide que XGBoost et 2.8x plus rapide que HistGB

Budget : 30 essais Optuna, 3 plis stratifiés pendant la recherche, MedianPruner
On estime à 5 minutes, au pire 9 minutes (calcul : temps mesuré (0.6) × nombre d'arbres (est. 1000 => 10) × essais (30) × plis (3))

| Hyperparamètre    | Plage proposée            | Ce qu'il règle    |
| ----------------- | -----------------         | ----------------- | 
| n_estimators      | 100 → 1000                | nombre d'arbres dans la chaîne |
| learning_rate     | 0,01 → 0,3 (échelle log)  | la part de correction apportée par chaque arbre. Petit = prudent mais il faut beaucoup d'arbres.|
| num_leaves        | 15 → 127                  | la complexité de chaque arbre (nombre de feuilles). Trop grand = risque de sur-apprentissage. |
| min_child_samples	| 10 → 100                  | le nombre minimum de produits dans une feuille. Plus grand = l'arbre ne crée pas de règle pour 3 produits isolés.|


In [12]:
import time
import optuna
from sklearn.metrics import f1_score

cv_search = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def objective(trial: optuna.Trial) -> float:
    """Entraîne LightGBM avec les hyperparamètres proposés par Optuna et renvoie son F1 macro moyen."""
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
    }
    model = Pipeline([("prep", prep_pipeline),
                      ("model", LGBMClassifier(random_state=42, verbose=-1, **params))])

    scores = []
    for step, (train_idx, val_idx) in enumerate(cv_search.split(X_train, y_train)):
        model.fit(X_train.iloc[train_idx], y_train[train_idx])
        pred = model.predict(X_train.iloc[val_idx])
        scores.append(f1_score(y_train[val_idx], pred, average="macro"))

        trial.report(np.mean(scores), step)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(scores)


In [13]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5),
)

start = time.perf_counter()
study.optimize(objective, n_trials=30)
elapsed = time.perf_counter() - start

print(f"Temps réel : {elapsed / 60:.1f} min (estimé : 5 min)")
print("Meilleur F1 (3 plis) :", round(study.best_value, 3))
print("Meilleurs paramètres :", study.best_params)


[I 2026-11-24 15:12:55,498] A new study created in memory with name: no-name-80025163-c35d-401d-a8da-bb4c46613eb5
[I 2026-11-24 15:13:08,356] Trial 0 finished with value: 0.8825335679872444 and parameters: {'n_estimators': 437, 'learning_rate': 0.2536999076681772, 'num_leaves': 97, 'min_child_samples': 64}. Best is trial 0 with value: 0.8825335679872444.
[I 2026-11-24 15:13:12,679] Trial 1 finished with value: 0.83995220672847 and parameters: {'n_estimators': 240, 'learning_rate': 0.01699897838270077, 'num_leaves': 21, 'min_child_samples': 88}. Best is trial 0 with value: 0.8825335679872444.
[I 2026-11-24 15:13:20,798] Trial 2 finished with value: 0.8850635463048205 and parameters: {'n_estimators': 641, 'learning_rate': 0.11114989443094977, 'num_leaves': 17, 'min_child_samples': 98}. Best is trial 2 with value: 0.8850635463048205.
[I 2026-11-24 15:13:39,961] Trial 3 finished with value: 0.8895820992933259 and parameters: {'n_estimators': 850, 'learning_rate': 0.020589728197687916, 'num

Temps réel : 10.5 min (estimé : 5 min)
Meilleur F1 (3 plis) : 0.891
Meilleurs paramètres : {'n_estimators': 715, 'learning_rate': 0.035549945040521405, 'num_leaves': 70, 'min_child_samples': 12}


Temps réel 10,9 min contre 5 estimées. L'estimation ne tenait compte que du nombre d'arbres ; elle oubliait que num_leaves augmente la taille de chaque arbre. Le budget reste sous les 15 minutes.

In [14]:
lgbm_tuned = Pipeline([("prep", prep_pipeline),
                       ("model", LGBMClassifier(random_state=42, verbose=-1, **study.best_params))])
results.append(evaluate_cv("lgbm_tuned", lgbm_tuned, X_train))

display(pd.DataFrame(results).set_index("model").round(3))


,f1_macro,f1_std,accuracy,balanced_accuracy,fit_time,score_time
model,,,,,,
logreg,0.767,0.008,0.805,0.766,0.727,0.017
rnd_forest,0.858,0.007,0.877,0.854,1.925,0.079
hist_gb,0.892,0.006,0.910,0.891,2.203,0.051
xg_boost,0.894,0.005,0.910,0.892,1.122,0.034
light_gbm,0.888,0.007,0.906,0.887,0.773,0.051
lgbm_tuned,0.895,0.010,0.912,0.894,11.612,0.406


In [15]:
res = pd.DataFrame(results).set_index("model")

gain = res.loc["lgbm_tuned", "f1_macro"] - res.loc["light_gbm", "f1_macro"]
threshold = 2 * max(res.loc["lgbm_tuned", "f1_std"], res.loc["light_gbm", "f1_std"])

print(f"Gain du réglage : {gain:+.3f} | seuil (2 × écart-type) : {threshold:.3f}")


Gain du réglage : +0.007 | seuil (2 × écart-type) : 0.020


Le réglage apporte +0,007, sous le seuil de 2 écarts-types (0,020) : le gain n'est pas démontré. Il multiplie le temps d'entraînement par 15 et celui de prédiction par 8. Le LightGBM par défaut reste le candidat.

## 3. Le tableau unique

Toutes les décisions sont prises sur la validation croisée. Modèle analysé à l'étape 4 : LightGBM par défaut. Le test est évalué une seule fois ci-dessous, pour tous les modèles, et aucun choix n'est modifié après. 

In [16]:
from sklearn.metrics import accuracy_score

test = df[df["split"] == "test"]
X_test = test[FEATURES]
y_test = label_encoder.transform(test["nutriscore_grade"])

models = {"logreg": logreg, "rnd_forest": rnd_forest, "hist_gb": hist_gb,
          "xg_boost": xg_boost, "light_gbm": light_gbm, "lgbm_tuned": lgbm_tuned}

test_rows = []
for name, model in models.items():
    start = time.perf_counter()
    model.fit(X_train, y_train)
    fit_s = time.perf_counter() - start

    start = time.perf_counter()
    pred = model.predict(X_test)
    predict_s = time.perf_counter() - start

    test_rows.append({
        "model": name,
        "f1_test": f1_score(y_test, pred, average="macro"),
        "accuracy_test": accuracy_score(y_test, pred),
        "fit_s": fit_s,
        "predict_ms_per_product": 1000 * predict_s / len(X_test),
    })

final = res[["f1_macro", "f1_std"]].join(pd.DataFrame(test_rows).set_index("model"))
final["gap_cv_test"] = final["f1_test"] - final["f1_macro"]
display(final.round(3))


,f1_macro,f1_std,f1_test,accuracy_test,fit_s,predict_ms_per_product,gap_cv_test
model,,,,,,,
logreg,0.767,0.008,0.772,0.808,1.038,0.002,0.005
rnd_forest,0.858,0.007,0.857,0.877,2.353,0.015,-0.001
hist_gb,0.892,0.006,0.891,0.907,1.978,0.009,-0.002
xg_boost,0.894,0.005,0.892,0.909,1.121,0.005,-0.001
light_gbm,0.888,0.007,0.886,0.904,0.686,0.007,-0.003
lgbm_tuned,0.895,0.010,0.893,0.909,7.471,0.058,-0.003


## 4. Confusions restantes